# 고성능 교사 학습 · Retinexformer · LOL-v2-real
**런타임 → 런타임 유형 변경 → GPU** 선택 후 위에서 아래로 실행하세요.
학생 증류에 쓸 고성능 교사(Retinexformer, 학습 파라미터 1,605,701개)를 **우리 학습 620쌍으로 처음부터** 학습합니다.
공개 사전학습 가중치는 쓰지 않습니다(우리 검증 분할이 포함된 데이터로 학습되었을 수 있음).

1. 짧은 점검이 반복당 시간과 **TOTAL_ITERS 전체의 예상 시간**을 출력합니다.
2. 예상 시간을 보고 설정 셀의 TOTAL_ITERS를 고친 뒤 **설정 셀을 다시 실행**하고 본 학습을 시작합니다. 각 실행 셀은 그 순간의 설정으로 명령을 새로 만듭니다.
3. 본 학습 시작 후에는 TOTAL_ITERS를 바꾸지 않습니다. 바꾸려면 EXPERIMENT도 새 이름으로 바꿉니다.
4. 끝나면 검증 69장 점수를 Max+ 일반 학습 결과와 비교합니다. **Max+보다 확실히 좋아야** 고성능 교사로 씁니다.

실행 코드는 GitHub 커밋 `1c2ad2128f75242559859b8ddc1eced21c3c88bf`로 고정됩니다.


In [ ]:
from pathlib import Path
TEACHER = "retinexformer"
EXPERIMENT = "teacher_v1"
SEED = 42
TOTAL_ITERS = 30000  # 점검 결과의 예상 시간을 보고 본 학습 전에 확정
ITERS_PER_SESSION = None  # 기본: 끝까지. 짧게 나눌 때만 숫자로(1000 단위 저장)
PILOT_ITERS = 100
BASE = Path("/content/drive/MyDrive/ondevice")
DATA_ZIP = BASE / "LOLv2.zip"
MAXPLUS_GT_SUMMARY = BASE / "runs" / "gt_random_v1" / "maxplus_gt_s42" / "summary.json"
CODE_REF = '1c2ad2128f75242559859b8ddc1eced21c3c88bf'


## 1. Drive 연결 및 실행 환경 준비


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import hashlib, json, os, shutil, subprocess, sys, uuid, zipfile
import torch
assert torch.cuda.is_available(), "GPU 런타임을 선택한 뒤 다시 실행하세요."
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def extract_checked(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError(f"잘못된 ZIP 경로: {name}")
        z.extractall(destination)

WORK = Path("/content/linet_gt")
WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK / f"source_{CODE_REF[:12]}"
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout",
                    "https://github.com/gidgogo/ondevice_project_v2.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_REF], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == CODE_REF, "코드 버전이 다릅니다. 새 런타임에서 다시 시작하세요."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(REPO / "scripts"))
print("실행 코드:", actual)
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
CONFIG = REPO / "configs/teacher_retinexformer.json"
from teacher_models import build_teacher

def teacher_plan():
    # 실행 셀마다 호출: 그 순간의 설정 셀 값으로 경로·명령을 새로 만든다.
    assert isinstance(TOTAL_ITERS, int) and TOTAL_ITERS > 0
    assert ITERS_PER_SESSION is None or (isinstance(ITERS_PER_SESSION, int) and ITERS_PER_SESSION > 0)
    settings = [f"teacher={json.dumps(TEACHER)}", f"seed={SEED}", f"total_iters={TOTAL_ITERS}"]
    effective = json.loads(CONFIG.read_text())
    effective.update(teacher=TEACHER, seed=SEED, total_iters=TOTAL_ITERS)
    command = [sys.executable, "-u", str(REPO / "scripts/train_teacher.py"),
               "--data-root", str(DATA_ROOT), "--manifests", str(MANIFESTS), "--config", str(CONFIG)]
    return {"settings": settings, "effective": effective, "command": command,
            "output": BASE / "teachers" / EXPERIMENT / f"{TEACHER}_s{SEED}"}

print("교사 파라미터:", sum(p.numel() for p in build_teacher(TEACHER).parameters()))
print("기본 설정 파일:", json.loads(CONFIG.read_text()))


## 2. 데이터 준비
학습 620쌍·검증 69쌍의 파일 해시를 확인합니다. 시험 이미지는 읽지 않습니다.


In [ ]:
assert DATA_ZIP.is_file(), f"기존 데이터 ZIP을 Drive에 올려주세요: {DATA_ZIP}"
LOCAL_ZIP = WORK / "LOLv2.zip"
DATA_FOLDER = WORK / "data"
DATA_MARKER = DATA_FOLDER / ".complete"
EXPECTED_DATA_SHA256 = "298e00fece107b64fb54769ae8bc5175b4f84a3f7e8134e651a0a4b5eb61198d"
if not DATA_MARKER.exists():
    shutil.copy2(DATA_ZIP, LOCAL_ZIP)
    assert file_hash(LOCAL_ZIP) == EXPECTED_DATA_SHA256, "기존 프로젝트의 LOLv2.zip과 파일이 다릅니다."
    extract_checked(LOCAL_ZIP, DATA_FOLDER)
    DATA_MARKER.write_text(EXPECTED_DATA_SHA256)
roots = [p for p in DATA_FOLDER.rglob("Real_captured") if (p / "Train/Low").is_dir()]
assert len(roots) == 1, f"Real_captured 폴더를 하나 찾을 수 있어야 합니다: {roots}"
DATA_ROOT = roots[0]
MANIFESTS = REPO / "manifests/lolv2_real"
verified = set()
for split, count in [("train", 620), ("validation", 69)]:
    manifest = json.loads((MANIFESTS / f"{split}.json").read_text())
    assert manifest["split"] == split and len(manifest["pairs"]) == count
    for row in manifest["pairs"]:
        for key in ("low", "high"):
            path = DATA_ROOT / row[key]
            if path not in verified:
                assert file_hash(path) == row[f"{key}_sha256"], f"데이터 불일치: {path}"
                verified.add(path)
    print(split, count, "쌍 확인 완료")


## 3. 짧은 실행 점검과 예상 시간
PILOT_ITERS만 학습하고 검증 69장을 한 번 실행합니다. 결과는 pilots 폴더에 따로 저장합니다.


In [ ]:
plan = teacher_plan()
pilot_dir = BASE / "pilots" / EXPERIMENT / f"{TEACHER}_{uuid.uuid4().hex[:8]}"
subprocess.run(plan["command"] + ["--output", str(pilot_dir), "--pilot-iters", str(PILOT_ITERS),
                                  "--set", *plan["settings"]], check=True, cwd=REPO / "scripts")
timing = json.loads((pilot_dir / "pilot_timing.json").read_text())
print(f"반복당 {timing['seconds_per_iter']:.3f}초 · TOTAL_ITERS={TOTAL_ITERS} 예상 {timing['estimated_hours_for_total_iters']:.1f}시간 ({timing['gpu']})")
print("TOTAL_ITERS를 바꾸려면 설정 셀을 고쳐 다시 실행한 뒤 본 학습 셀로 넘어가세요.")


## 4. 본 학습 / 이어서 학습
1000회마다 검증·저장합니다. 끊기면 준비 셀부터 다시 실행하고 이 셀을 실행하면 마지막 저장 지점부터 이어갑니다. 저장 직후 끊겨 결과 파일이 빠졌으면 이어서 실행할 때 checkpoint에서 복원합니다.


In [ ]:
plan = teacher_plan()
OUTPUT = plan["output"]
resumable = any((OUTPUT / name).exists() for name in ("last.pt", "last.prev.pt"))
if OUTPUT.exists() and any(OUTPUT.iterdir()) and not resumable:
    raise RuntimeError("복구할 checkpoint가 없습니다. EXPERIMENT를 새 이름으로 바꾸세요.")
if (OUTPUT / "run.json").exists():
    assert json.loads((OUTPUT / "run.json").read_text())["config"] == plan["effective"], "기존 실험과 설정이 다릅니다(TOTAL_ITERS 등). 원래 값으로 되돌리거나 새 EXPERIMENT 이름을 사용하세요."
cmd = plan["command"] + ["--output", str(OUTPUT)]
if ITERS_PER_SESSION is not None:
    cmd += ["--iters-per-session", str(ITERS_PER_SESSION)]
if resumable:
    cmd += ["--resume"]
print("실행 설정:", plan["settings"], "→", OUTPUT)
subprocess.run(cmd + ["--set", *plan["settings"]], check=True, cwd=REPO / "scripts")
summary = json.loads((OUTPUT / "summary.json").read_text())
print(f"저장 완료: {summary['completed_iters']}/{summary['target_iters']} · 최고 검증 {summary['best']}")


## 5. 검증 69장 평가와 Max+ 일반 학습 비교
고성능 교사로 쓰려면 검증 PSNR이 Max+ 일반 학습보다 확실히 높아야 합니다. 시험 분할은 평가하지 않습니다.


In [ ]:
OUTPUT = teacher_plan()["output"]
assert json.loads((OUTPUT / "summary.json").read_text())["training_complete"], "교사 학습이 아직 끝나지 않았습니다."
REPORT = OUTPUT / "validation_report.json"
subprocess.run([sys.executable, str(REPO / "scripts/eval_checkpoint.py"), "--model", TEACHER,
                "--checkpoint", str(OUTPUT / "best.pt"), "--data-root", str(DATA_ROOT),
                "--manifests", str(MANIFESTS), "--device", "cuda", "--output", str(REPORT)],
               check=True, cwd=REPO / "scripts")
teacher = json.loads(REPORT.read_text())["mean"]
print("교사 검증:", teacher)
if MAXPLUS_GT_SUMMARY.exists():
    maxplus = json.loads(MAXPLUS_GT_SUMMARY.read_text())["best"]
    print(f"교사 - Max+ 일반 학습: PSNR {teacher['psnr'] - maxplus['psnr']:+.2f} dB, SSIM {teacher['ssim'] - maxplus['ssim']:+.4f}")
else:
    print("Max+ 일반 학습 결과가 이 Drive에 없습니다. summary.json을 받아 MAXPLUS_GT_SUMMARY 경로에 두세요.")
print("증류에 쓸 교사 checkpoint:", OUTPUT / "best.pt")
